# TP final · Opción B · Efecto de la música

Problema: predecir `Music effects` (`Improve`, `No effect`, `Worsen`) a partir de los hábitos de escucha. Es clasificación. La métrica que va a decidir es el F1 macro de test.

Esta parte solo mira los datos. No imputa, no descarta filas y no entrena.

Fuente: [Music & Mental Health Survey Results](https://www.kaggle.com/datasets/catherinerasgaitis/mxmh-survey-results), Catherine Rasgaitis, licencia CC0. El CSV está en `data/mxmh_survey_results.csv`.


## Cómo se eligen los puntajes de salud

`Anxiety`, `Depression`, `Insomnia` y `OCD` pueden entrar como variables. El widget `puntajes_salud` elige el grupo:

- `ninguna`: no entra ningún puntaje.
- `algunas`: entran solo las columnas escritas en `puntajes_algunas`, separadas por coma.
- `todas`: entran los cuatro.

El corte de modelos usa esa elección. Esta sección todavía no entrena: imprime la lista para comprobarla.


In [ ]:
dbutils.widgets.text("csv_path", "", "Ruta CSV (vacio = junto al notebook)")
dbutils.widgets.dropdown("puntajes_salud", "ninguna", ["ninguna", "algunas", "todas"], "Puntajes de salud")
dbutils.widgets.text("puntajes_algunas", "Depression", "Si algunas: columnas separadas por coma")


In [ ]:
from pyspark.sql import functions as F

SALUD = ["Anxiety", "Depression", "Insomnia", "OCD"]

def resolver_csv():
    pedido = dbutils.widgets.get("csv_path").strip()
    if pedido:
        return pedido
    notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    carpeta = "/Workspace" + notebook.rsplit("/", 1)[0]
    return "file:" + carpeta + "/data/mxmh_survey_results.csv"

csv_path = resolver_csv()
print("csv:", csv_path)

crudo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)
print(f"filas: {crudo.count():,}  columnas: {len(crudo.columns)}")


In [ ]:
def columnas_salud():
    modo = dbutils.widgets.get("puntajes_salud")
    if modo == "ninguna":
        return []
    if modo == "todas":
        return list(SALUD)
    pedidas = [parte.strip() for parte in dbutils.widgets.get("puntajes_algunas").split(",") if parte.strip()]
    desconocidas = [nombre for nombre in pedidas if nombre not in SALUD]
    if desconocidas:
        raise ValueError(f"No son puntajes de salud: {desconocidas}. Opciones: {SALUD}")
    return pedidas

elegidas = columnas_salud()
print("modo:", dbutils.widgets.get("puntajes_salud"))
print("puntajes que entran:", elegidas if elegidas else "(ninguno)")


## 1. Qué hay en la tabla

Conteos, nulos y las tres clases. Una celda vacía cuenta como faltante, igual que un nulo. No se rellena nada.


In [ ]:
display(crudo.limit(5))
print("columnas:")
for nombre in crudo.columns:
    print("-", nombre)


In [ ]:
faltantes = crudo.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)).alias(c)
    for c in crudo.columns
])
pares = ", ".join("'" + c.replace("'", "") + "', `" + c + "`" for c in crudo.columns)
largo = faltantes.select(
    F.expr(f"stack({len(crudo.columns)}, {pares}) as (columna, faltantes)")
).orderBy(F.desc("faltantes"), "columna")
display(largo)


In [ ]:
etiqueta = F.when(
    F.col("Music effects").isNull() | (F.trim(F.col("Music effects")) == ""),
    "(vacio)",
).otherwise(F.col("Music effects"))
display(
    crudo.groupBy(etiqueta.alias("Music effects"))
    .count()
    .orderBy(F.desc("count"))
)


## 2. Age, horas y BPM

Resumen numérico y las filas de BPM que no son un tempo usable (vacío, no numérico, o fuera de 40–250). Sirve para decidir el arreglo. Esta celda no modifica la tabla.


In [ ]:
bpm = F.col("BPM").cast("double")
display(crudo.select(
    F.count("*").alias("filas"),
    F.count(F.when(F.col("Age").isNull(), 1)).alias("age_vacio"),
    F.min("Age").alias("age_min"),
    F.max("Age").alias("age_max"),
    F.count(F.when(F.col("Hours per day").isNull(), 1)).alias("horas_vacio"),
    F.min("Hours per day").alias("horas_min"),
    F.max("Hours per day").alias("horas_max"),
    F.count(F.when(F.col("BPM").isNull() | (F.trim(F.col("BPM").cast("string")) == ""), 1)).alias("bpm_vacio"),
    F.count(F.when(F.col("BPM").isNotNull() & (F.trim(F.col("BPM").cast("string")) != "") & bpm.isNull(), 1)).alias("bpm_no_numerico"),
    F.min(bpm).alias("bpm_min"),
    F.max(bpm).alias("bpm_max"),
))


In [ ]:
raros = crudo.where(
    F.col("BPM").isNull()
    | (F.trim(F.col("BPM").cast("string")) == "")
    | bpm.isNull()
    | (bpm < 40)
    | (bpm > 250)
)
print(f"filas con BPM vacio, no numerico o fuera de 40-250: {raros.count()}")
display(raros.select("Age", "Fav genre", "BPM", "Music effects").orderBy(bpm.asc_nulls_first()))


## Modelos previstos

Cuando el arreglo esté decidido, los cuatro ven la misma matriz y el mismo corte 80/20 estratificado (`random_state=7`). Los tres de árboles usan el balanceo de clases. Gana el mayor F1 macro de test.

| Run | Modelo |
|---|---|
| `logistica` | Regresión logística, `C=1` |
| `random_forest` | Random forest |
| `lightgbm` | LightGBM |
| `catboost` | CatBoost |

El entrenamiento queda para después de mirar esta exploración. No se imputa en este notebook todavía.
